In [16]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob, shutil
sys.path.append("/content/drive/MyDrive/tcc")
from src import config

import numpy as np
import pandas as pd

ZIP = f"{config.BRUTOS}/synthbuster/synthbuster.zip"
SB = "/content/synthbuster/synthbuster"
if not os.path.exists(SB):
    !unzip -q "{ZIP}" -d /content/synthbuster
print(sorted(os.listdir(SB)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['.directory', 'LICENCE.txt', 'dalle2', 'dalle3', 'firefly', 'glide', 'midjourney-v5', 'prompts.csv', 'readme.md', 'stable-diffusion-1-3', 'stable-diffusion-1-4', 'stable-diffusion-2', 'stable-diffusion-xl']


In [17]:
raise_csv = pd.read_csv(config.RAISE_CSV)

ids = sorted(raise_csv["File"].tolist())
rng = np.random.default_rng(config.SEED)
ids_embaralhados = rng.permutation(ids)

geradores = config.GERADORES_TREINO
gerador_do_par = {pid: geradores[i % len(geradores)] for i, pid in enumerate(ids_embaralhados)}

def achar(gerador, pid):
    encontrados = glob.glob(f"{SB}/{gerador}/{pid}.*")
    return encontrados[0] if encontrados else None

linhas = []
for pid in ids:
    g = gerador_do_par[pid]
    linhas.append({
        "id": pid,
        "camera": raise_csv.loc[raise_csv["File"] == pid, "Device"].iloc[0],
        "gerador": g,
        "origem_ia": achar(g, pid),
        "origem_firefly": achar("firefly", pid),
    })

pares = pd.DataFrame(linhas)
print("Pares:", len(pares))
print("Sem imagem de IA:", pares["origem_ia"].isna().sum())
print("Sem imagem do Firefly:", pares["origem_firefly"].isna().sum())
print(pares["gerador"].value_counts())
print(pares["camera"].value_counts())

Pares: 1000
Sem imagem de IA: 0
Sem imagem do Firefly: 0
gerador
midjourney-v5           143
stable-diffusion-1-4    143
stable-diffusion-2      143
dalle2                  143
dalle3                  143
stable-diffusion-1-3    143
stable-diffusion-xl     142
Name: count, dtype: int64
camera
Nikon D7000    729
Nikon D90      263
Nikon D40        8
Name: count, dtype: int64


In [18]:
DEST = f"{config.PADRONIZADOS}/original"

for _, p in pares.iterrows():
    destino_ia = f"{DEST}/ia/{p['id']}_{p['gerador']}.png"
    if p["origem_ia"] and not os.path.exists(destino_ia):
        shutil.copy(p["origem_ia"], destino_ia)

    if p["origem_firefly"]:
        ext = os.path.splitext(p["origem_firefly"])[1]
        destino_ff = f"{DEST}/firefly/{p['id']}_firefly{ext}"
        if not os.path.exists(destino_ff):
            shutil.copy(p["origem_firefly"], destino_ff)

print("IA:", len(os.listdir(f"{DEST}/ia")))
print("Firefly:", len(os.listdir(f"{DEST}/firefly")))

IA: 1000
Firefly: 1000


In [22]:
pares_salvar = pares.copy()
pares_salvar["arquivo_ia"] = "dados_padronizados/original/ia/" + pares_salvar["id"] + "_" + pares_salvar["gerador"] + ".png"
pares_salvar["arquivo_real"] = "dados_padronizados/original/real/" + pares_salvar["id"] + ".png"

def caminho_firefly(origem):
    if not isinstance(origem, str):
        return None
    base, ext = os.path.splitext(os.path.basename(origem))
    return f"dados_padronizados/original/firefly/{base}_firefly{ext}"

pares_salvar["arquivo_firefly"] = pares_salvar["origem_firefly"].map(caminho_firefly)
pares_salvar = pares_salvar.drop(columns=["origem_ia", "origem_firefly"])

pares_salvar.to_csv(config.PARES_CSV, index=False)
print("Salvo?", os.path.exists(config.PARES_CSV))
print(pd.read_csv(config.PARES_CSV).head(3).to_string())

Salvo? True
           id       camera               gerador                                                          arquivo_ia                                     arquivo_real                                             arquivo_firefly
0  r000da54ft    Nikon D90         midjourney-v5         dados_padronizados/original/ia/r000da54ft_midjourney-v5.png  dados_padronizados/original/real/r000da54ft.png  dados_padronizados/original/firefly/r000da54ft_firefly.png
1  r001d260dt  Nikon D7000  stable-diffusion-1-4  dados_padronizados/original/ia/r001d260dt_stable-diffusion-1-4.png  dados_padronizados/original/real/r001d260dt.png  dados_padronizados/original/firefly/r001d260dt_firefly.png
2  r002fc3e2t  Nikon D7000  stable-diffusion-1-4  dados_padronizados/original/ia/r002fc3e2t_stable-diffusion-1-4.png  dados_padronizados/original/real/r002fc3e2t.png  dados_padronizados/original/firefly/r002fc3e2t_firefly.png


In [23]:
DEST = f"{config.PADRONIZADOS}/original"
for pasta in ["real", "ia", "firefly"]:
    print(pasta, len(os.listdir(f"{DEST}/{pasta}")))

real 1000
ia 1000
firefly 1000


In [24]:
pares_salvar = pares.copy()
pares_salvar["arquivo_ia"] = "dados_padronizados/original/ia/" + pares_salvar["id"] + "_" + pares_salvar["gerador"] + ".png"
pares_salvar["arquivo_real"] = "dados_padronizados/original/real/" + pares_salvar["id"] + ".png"

def caminho_firefly(origem):
    if not isinstance(origem, str):
        return None
    base, ext = os.path.splitext(os.path.basename(origem))
    return f"dados_padronizados/original/firefly/{base}_firefly{ext}"

pares_salvar["arquivo_firefly"] = pares_salvar["origem_firefly"].map(caminho_firefly)
pares_salvar = pares_salvar.drop(columns=["origem_ia", "origem_firefly"])

pares_salvar.to_csv(config.PARES_CSV, index=False)
print("Salvo?", os.path.exists(config.PARES_CSV))
print(pd.read_csv(config.PARES_CSV).head(3).to_string())

Salvo? True
           id       camera               gerador                                                          arquivo_ia                                     arquivo_real                                             arquivo_firefly
0  r000da54ft    Nikon D90         midjourney-v5         dados_padronizados/original/ia/r000da54ft_midjourney-v5.png  dados_padronizados/original/real/r000da54ft.png  dados_padronizados/original/firefly/r000da54ft_firefly.png
1  r001d260dt  Nikon D7000  stable-diffusion-1-4  dados_padronizados/original/ia/r001d260dt_stable-diffusion-1-4.png  dados_padronizados/original/real/r001d260dt.png  dados_padronizados/original/firefly/r001d260dt_firefly.png
2  r002fc3e2t  Nikon D7000  stable-diffusion-1-4  dados_padronizados/original/ia/r002fc3e2t_stable-diffusion-1-4.png  dados_padronizados/original/real/r002fc3e2t.png  dados_padronizados/original/firefly/r002fc3e2t_firefly.png


In [25]:
from PIL import Image
from tqdm import tqdm

for classe in ["real", "ia"]:
    os.makedirs(f"{config.PADRONIZADOS}/jpeg90/{classe}", exist_ok=True)
    for nome in tqdm(sorted(os.listdir(f"{DEST}/{classe}")), desc=classe):
        saida = f"{config.PADRONIZADOS}/jpeg90/{classe}/{nome.rsplit('.', 1)[0]}.jpg"
        if os.path.exists(saida):
            continue
        with Image.open(f"{DEST}/{classe}/{nome}") as img:
            img.convert("RGB").save(saida, "JPEG", quality=config.JPEG_CONTROLE, subsampling=2)

ia: 100%|██████████| 1000/1000 [00:00<00:00, 3576.49it/s]


In [26]:
pares = pd.read_csv(config.PARES_CSV)
linhas = []

for _, p in pares.iterrows():
    pid = p["id"]
    ia_nome = f"{pid}_{p['gerador']}"
    linhas += [
        {"id": pid, "classe": "real", "fonte": "raise", "aparelho_ou_gerador": p["camera"],
         "versao": "original", "papel": "principal",
         "arquivo": f"dados_padronizados/original/real/{pid}.png"},
        {"id": pid, "classe": "ia", "fonte": "synthbuster", "aparelho_ou_gerador": p["gerador"],
         "versao": "original", "papel": "principal",
         "arquivo": f"dados_padronizados/original/ia/{ia_nome}.png"},
        {"id": pid, "classe": "real", "fonte": "raise", "aparelho_ou_gerador": p["camera"],
         "versao": "jpeg90", "papel": "principal",
         "arquivo": f"dados_padronizados/jpeg90/real/{pid}.jpg"},
        {"id": pid, "classe": "ia", "fonte": "synthbuster", "aparelho_ou_gerador": p["gerador"],
         "versao": "jpeg90", "papel": "principal",
         "arquivo": f"dados_padronizados/jpeg90/ia/{ia_nome}.jpg"},
        {"id": pid, "classe": "ia", "fonte": "synthbuster", "aparelho_ou_gerador": "firefly",
         "versao": "original", "papel": "ood",
         "arquivo": p["arquivo_firefly"]},
    ]

df = pd.DataFrame(linhas)
df["existe"] = df["arquivo"].map(lambda a: os.path.exists(f"{config.DRIVE}/{a}"))
print("Arquivos faltando:", (~df["existe"]).sum())

df.drop(columns="existe").to_csv(config.DATASET_CSV, index=False)
print(df.groupby(["papel", "versao", "classe"]).size())

Arquivos faltando: 0
papel      versao    classe
ood        original  ia        1000
principal  jpeg90    ia        1000
                     real      1000
           original  ia        1000
                     real      1000
dtype: int64


In [15]:
print(os.listdir(config.DRIVE))

['dados_padronizados', 'redes_sociais', 'dados_brutos', 'ela', 'modelos', 'resultados', 'notebooks', 'src']
